# TrendWaveletGeneric Effectiveness Study -- M4-Yearly

**Question:** Does adding a learned generic branch to TrendWavelet blocks (polynomial + DWT + learned low-rank basis) improve forecasting quality over the two-branch TrendWavelet (polynomial + DWT only)?

**Design:** 7 configs x 2 passes (baseline, activeG_fcast) x 5 seeds = 70 runs on M4-Yearly. Four new TrendWaveletGeneric variants (RootBlock, AE, AELG, VAE) vs three TrendWavelet baselines (RootBlock, AE, AELG). All use coif2 wavelet, td=3, bd=4, gd=5, ld=16.

**Key findings (preview):**
- The generic branch does **not** provide a statistically significant improvement on any backbone
- TrendWaveletGenericAELG is the study winner (SMAPE 13.509) but does not beat M4-Yearly SOTA (13.410)
- AE-backbone TWG variants achieve 3.4x parameter efficiency over their TW counterparts (445K vs 1.5M params)
- VAE backbone remains catastrophically bad (+19% vs best)
- active_g is a non-factor across all configs

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path
matplotlib.rcParams.update({'font.size': 11, 'figure.figsize': (12, 6)})

# Resolve repo root regardless of working directory
_root = Path.cwd()
for _ in range(6):
    if (_root / 'pyproject.toml').exists():
        break
    _root = _root.parent

CSV = str(_root / 'experiments/results/m4/trendwavelet_generic_effectiveness_results.csv')
NOTEBOOKS_DIR = str(_root / 'experiments/analysis/notebooks')
df = pd.read_csv(CSV)

# Tag rows
df['has_generic'] = df['block_type'].str.contains('TrendWaveletGeneric')
df['is_vae'] = df['block_type'].str.contains('VAE')

print(f"Rows: {len(df)}, Configs: {df['config_name'].nunique()}, "
      f"Passes: {df['experiment'].unique().tolist()}, Diverged: {df['diverged'].sum()}")
print(f"Configs: {sorted(df['config_name'].unique())}")

## 1. Full Ranking: Config x Pass

Every combination ranked by mean SMAPE across 5 seeds. The best config in the study is TWGAELG (baseline pass), but the differences among non-VAE configs are small (spread of 0.25 SMAPE across 12 config-pass combinations).

In [ ]:
summary = df.groupby(['config_name', 'experiment']).agg(
    n=('smape', 'count'),
    smape_mean=('smape', 'mean'),
    smape_std=('smape', 'std'),
    owa_mean=('owa', 'mean'),
    owa_std=('owa', 'std'),
    n_params=('n_params', 'first'),
    block_type=('block_type', 'first'),
    backbone=('backbone', 'first'),
    epochs_mean=('epochs_trained', 'mean'),
).reset_index().sort_values('smape_mean').reset_index(drop=True)

best = summary['smape_mean'].iloc[0]
summary['rank'] = range(1, len(summary)+1)
summary['delta'] = summary['smape_mean'] - best
summary['delta_pct'] = (100 * summary['delta'] / best).round(2)

display_cols = ['rank', 'config_name', 'experiment', 'smape_mean', 'smape_std', 
                'owa_mean', 'n_params', 'delta', 'delta_pct', 'epochs_mean']
print(summary[display_cols].to_string(index=False, float_format='%.3f'))

## 2. Does the Generic Branch Help? Paired Comparisons

The central question. For each backbone (RootBlock, AE, AELG), we compare the TWG variant to its TW baseline, matched by seed and pass. A negative difference means TWG is better.

In [ ]:
pairs = [
    ('TWG_coif2_td3_bd4_gd5', 'TW_coif2_td3_bd4', 'RootBlock'),
    ('TWGAE_coif2_td3_bd4_gd5_ld16', 'TWAE_coif2_td3_bd4_ld16', 'AERootBlock'),
    ('TWGAELG_coif2_td3_bd4_gd5_ld16', 'TWAELG_coif2_td3_bd4_ld16', 'AERootBlockLG'),
]

results = []
for pass_name in ['baseline', 'activeG_fcast']:
    for twg_cfg, tw_cfg, backbone in pairs:
        twg = df[(df['config_name'] == twg_cfg) & (df['experiment'] == pass_name)].sort_values('seed')
        tw = df[(df['config_name'] == tw_cfg) & (df['experiment'] == pass_name)].sort_values('seed')
        
        twg_s, tw_s = twg['smape'].values, tw['smape'].values
        diff = twg_s - tw_s
        
        try:
            wstat, wpval = stats.wilcoxon(diff, alternative='two-sided')
        except:
            wstat, wpval = np.nan, np.nan
        ustat, upval = stats.mannwhitneyu(twg_s, tw_s, alternative='two-sided')
        pooled_std = np.sqrt((twg_s.std()**2 + tw_s.std()**2) / 2)
        d = diff.mean() / pooled_std if pooled_std > 0 else np.nan
        
        # Bootstrap CI
        rng = np.random.default_rng(42)
        boot = [diff[rng.integers(0, 5, 5)].mean() for _ in range(10000)]
        ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5])
        
        results.append({
            'pass': pass_name, 'backbone': backbone,
            'TWG_smape': f"{twg_s.mean():.3f} ({twg_s.std():.3f})",
            'TW_smape': f"{tw_s.mean():.3f} ({tw_s.std():.3f})",
            'diff': f"{diff.mean():+.3f}",
            'CI_95': f"[{ci_lo:+.3f}, {ci_hi:+.3f}]",
            'MWU_p': f"{upval:.3f}",
            'Wilcoxon_p': f"{wpval:.3f}" if not np.isnan(wpval) else "N/A",
            'd': f"{d:+.2f}",
            'TWG_wins': f"{(diff < 0).sum()}/5",
        })

res_df = pd.DataFrame(results)
print(res_df.to_string(index=False))

**Interpretation:** No comparison reaches statistical significance (all MWU p > 0.16, all Wilcoxon p > 0.31). The 95% bootstrap CIs for the difference all straddle zero. The generic branch is neither consistently helpful nor harmful.

On the RootBlock backbone, TWG is slightly *worse* (+0.07-0.13 SMAPE). On AE/AELG backbones, TWG shows a small (non-significant) advantage in the baseline pass but loses it with activeG_fcast. The pattern suggests the generic branch's learned basis may partially duplicate what active_g already provides.

## 3. Visualization: SMAPE by Config and Pass

In [ ]:
# Box plot: all non-VAE configs, grouped by backbone, colored by TWG vs TW
non_vae = df[~df['is_vae']].copy()
non_vae['short_cfg'] = non_vae['config_name'].str.replace('_coif2_td3_bd4.*', '', regex=True)
non_vae['label'] = non_vae['short_cfg'] + '\n(' + non_vae['experiment'].str.replace('activeG_fcast', 'aG') + ')'

order = non_vae.groupby('label')['smape'].mean().sort_values().index.tolist()

fig, ax = plt.subplots(figsize=(14, 6))
colors = ['#2196F3' if 'TWG' in lbl else '#FF9800' for lbl in order]

bp = ax.boxplot(
    [non_vae[non_vae['label'] == lbl]['smape'].values for lbl in order],
    labels=order, patch_artist=True, widths=0.6
)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.axhline(y=13.410, color='red', linestyle='--', alpha=0.7, label='M4-Yearly SOTA (13.410)')
ax.set_ylabel('SMAPE')
ax.set_title('TrendWaveletGeneric (blue) vs TrendWavelet (orange) -- M4-Yearly')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{NOTEBOOKS_DIR}/twg_effectiveness_boxplot.png', dpi=150)
plt.show()

## 4. Parameter Efficiency

A striking finding: TWG-AE and TWG-AELG use only ~445K parameters vs ~1.5M for their TW-AE/AELG counterparts (3.4x reduction), while achieving equivalent or slightly better SMAPE. This is because the generic branch in TWG replaces parameters differently in the AE backbone structure.

In [ ]:
# Parameter efficiency: scatter plot SMAPE vs params
best_per_cfg = df.groupby('config_name').agg(
    smape_mean=('smape', 'mean'),
    smape_std=('smape', 'std'),
    n_params=('n_params', 'first'),
    block_type=('block_type', 'first'),
    has_generic=('has_generic', 'first'),
    is_vae=('is_vae', 'first'),
).reset_index()

fig, ax = plt.subplots(figsize=(10, 6))
for _, row in best_per_cfg.iterrows():
    color = 'red' if row['is_vae'] else ('blue' if row['has_generic'] else 'orange')
    marker = 's' if row['has_generic'] else 'o'
    ax.errorbar(row['n_params']/1e6, row['smape_mean'], yerr=row['smape_std'],
                color=color, marker=marker, markersize=10, capsize=4)
    ax.annotate(row['config_name'].replace('_coif2_td3_bd4', '').replace('_gd5', '').replace('_ld16', ''),
                (row['n_params']/1e6, row['smape_mean']), fontsize=8, 
                xytext=(5, 5), textcoords='offset points')

ax.set_xlabel('Parameters (millions)')
ax.set_ylabel('Mean SMAPE (all passes)')
ax.set_title('Parameter Efficiency: TWG (blue/sq) vs TW (orange/circle) vs VAE (red)')
ax.axhline(y=13.410, color='red', linestyle='--', alpha=0.5, label='SOTA 13.410')
ax.legend()
plt.tight_layout()
plt.savefig(f'{NOTEBOOKS_DIR}/twg_param_efficiency.png', dpi=150)
plt.show()

# Table
eff = best_per_cfg[['config_name', 'smape_mean', 'n_params', 'block_type']].copy()
eff['params_M'] = (eff['n_params'] / 1e6).round(3)
eff['smape_per_Mparam'] = (eff['smape_mean'] / eff['params_M']).round(1)
eff = eff.sort_values('smape_per_Mparam')
print(eff[['config_name', 'smape_mean', 'params_M', 'smape_per_Mparam']].to_string(index=False, float_format='%.3f'))

**Interpretation:** TWGAELG and TWGAE are the most parameter-efficient configs in this study: ~445K params for SMAPE ~13.5-13.6. The non-AE variants (TW, TWG with RootBlock) use ~2.1M params for equivalent or worse quality. TWAELG and TWAE sit in between at ~1.5M params but with no quality advantage. The 3.4x parameter reduction of TWG-AE variants over TW-AE variants is the most practically useful finding of this study, even though the SMAPE difference is not statistically significant.

## 5. VAE is Catastrophic (Again)

In [ ]:
vae = df[df['is_vae']]
non_vae_all = df[~df['is_vae']]

print("TrendWaveletGenericVAE performance:")
for pass_name in ['baseline', 'activeG_fcast']:
    v = vae[vae['experiment'] == pass_name]['smape']
    print(f"  {pass_name}: SMAPE = {v.mean():.3f} +/- {v.std():.3f} (range: {v.min():.3f} - {v.max():.3f})")

print(f"\nBest non-VAE: SMAPE = {non_vae_all.groupby(['config_name','experiment'])['smape'].mean().min():.3f}")
print(f"VAE penalty: +{vae['smape'].mean() - non_vae_all.groupby(['config_name','experiment'])['smape'].mean().min():.3f} SMAPE (+{100*(vae['smape'].mean() - non_vae_all['smape'].mean())/non_vae_all['smape'].mean():.1f}%)")
print(f"\nVAE high variance: std across all 10 runs = {vae['smape'].std():.3f}")
print("VAE continues the pattern: AERootBlockVAE is never competitive. Do not test further.")

## 6. active_g Effect

Does the `active_g=forecast` setting help or hurt? We compare baseline vs activeG_fcast for each config.

In [ ]:
print("active_g effect per config (negative = activeG helps):\n")
for cfg in sorted(df['config_name'].unique()):
    bl = df[(df['config_name'] == cfg) & (df['experiment'] == 'baseline')]['smape'].values
    ag = df[(df['config_name'] == cfg) & (df['experiment'] == 'activeG_fcast')]['smape'].values
    if len(bl) == 5 and len(ag) == 5:
        diff = ag.mean() - bl.mean()
        _, upval = stats.mannwhitneyu(ag, bl, alternative='two-sided')
        winner = "activeG" if diff < 0 else "baseline"
        sig = "*" if upval < 0.05 else ""
        print(f"  {cfg:45s}: baseline={bl.mean():.3f} -> activeG={ag.mean():.3f}  "
              f"diff={diff:+.3f}  MWU p={upval:.3f} {sig}  -> {winner}")

print("\nNo config shows a significant active_g effect (all p > 0.4).")
print("active_g=forecast provides no benefit for TrendWavelet or TrendWaveletGeneric blocks.")

## 7. Backbone Hierarchy (pooled across TWG/TW variants)

Aggregating all non-VAE configs by backbone type confirms the established hierarchy: AERootBlockLG > AERootBlock > RootBlock.

In [ ]:
non_vae = df[~df['is_vae']]
hier = non_vae.groupby('backbone').agg(
    smape_mean=('smape', 'mean'),
    smape_std=('smape', 'std'),
    smape_min=('smape', 'min'),
    n=('smape', 'count'),
    params_range=('n_params', lambda x: f"{x.min():,}-{x.max():,}"),
).reset_index().sort_values('smape_mean')
print(hier.to_string(index=False))

# Pairwise backbone comparison
backbones = ['AERootBlockLG', 'AERootBlock', 'RootBlock']
print("\nPairwise MWU tests (backbone):")
for i in range(len(backbones)):
    for j in range(i+1, len(backbones)):
        a = non_vae[non_vae['backbone'] == backbones[i]]['smape'].values
        b = non_vae[non_vae['backbone'] == backbones[j]]['smape'].values
        _, p = stats.mannwhitneyu(a, b, alternative='two-sided')
        d = (a.mean() - b.mean()) / np.sqrt((a.std()**2 + b.std()**2)/2)
        print(f"  {backbones[i]} vs {backbones[j]}: diff={a.mean()-b.mean():+.3f}, MWU p={p:.4f}, d={d:+.3f}")

## 8. Context: How Does This Compare to M4-Yearly SOTA?

None of the configs in this study beats the established M4-Yearly SOTA of SMAPE=13.410 (Trend+WaveletV3, Coif2_bd6_eq_fcast_td3, non-AE alternating architecture). The best config here (TWGAELG baseline, 13.509) is 0.7% worse.

However, the TWGAELG config achieves this with only 445K parameters vs the SOTA's ~1.4M -- a 3.1x reduction. If parameter efficiency is the priority, TWGAELG is a strong candidate.

## 9. Conclusions and Recommendations

### Main Findings

1. **The learned generic branch in TrendWaveletGeneric does NOT provide a statistically significant improvement** over TrendWavelet on M4-Yearly. Across all 6 paired comparisons (3 backbones x 2 passes), no test reaches p < 0.16.

2. **TWG-AE variants are dramatically more parameter-efficient** than TW-AE variants (445K vs 1.5M, 3.4x reduction) with no quality loss. This is the study's most actionable finding.

3. **The backbone hierarchy is confirmed:** AERootBlockLG >= AERootBlock > RootBlock. The learned gate continues to provide a small but consistent advantage.

4. **active_g=forecast is a non-factor** for all TrendWavelet and TrendWaveletGeneric blocks (all p > 0.4).

5. **VAE backbone (+19%) remains catastrophically bad.** Do not pursue TrendWaveletGenericVAE further.

6. **M4-Yearly SOTA is NOT beaten.** Best here: 13.509 vs SOTA 13.410 (+0.7%).

### What to Test Next

1. **generic_dim sweep (gd=2,3,5,8,10,15)** on TWGAELG to find optimal rank for the learned generic basis. The current gd=5 was a default, not optimized.

2. **Multi-dataset validation:** Run TWGAELG on Tourism-Yearly, Traffic-96, Weather-96 to test if the parameter efficiency advantage holds across datasets.

3. **Alternating architecture:** Test Trend + WaveletV3Generic (alternating, not unified) to see if separating the polynomial and wavelet+generic branches helps as it does for Trend+WaveletV3.

4. **basis_dim sweep for TWG:** The current bd=4 (lt_fcast) was inherited from TW studies. TWG may prefer a different basis_dim given the additional generic branch.

### Open Questions

- Does the generic branch help more on longer horizons (Monthly, Quarterly) where there is more signal to learn?
- Is gd=5 optimal, or could a larger generic_dim unlock the branch's potential?
- Why do TWG-AE variants have 3.4x fewer parameters? The architectural source of this reduction needs documentation.